# 🎬 Netflix Recommendation System using Embeddings

## 📌 Objective
Build a content-based recommendation system for Netflix titles using semantic embeddings.

## 🧠 Why Embeddings?
Unlike TF-IDF, which relies on exact word overlap, embeddings capture semantic meaning.
This helps produce more relevant recommendations when two titles are similar in theme, even if they use different words.

## 📥 Input
A Netflix title

## 📤 Output
Top 5 similar Netflix titles

## Imports

In [38]:
# Data manipulation
import pandas as pd
import numpy as np

# Embedding model
from sentence_transformers import SentenceTransformer

# Similarity computation
from sklearn.metrics.pairwise import cosine_similarity

## Load Data
We load the Netflix dataset and inspect the first rows.

In [39]:
df = pd.read_csv("./data/netflix_cleaned.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,year_added,month_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",2021.0,9.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",2021.0,9.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,2021.0,9.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",2021.0,9.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,2021.0,9.0


## Data Preparation

We select only the columns that are useful for recommendation:
- title
- listed_in
- description
- director
- cast
- type

In [40]:
rec_df = df[["title", "listed_in", "description", "director", "cast", "type"]].copy()
rec_df.head()

,title,listed_in,description,director,cast,type
0,Dick Johnson Is Dead,Documentaries,"As her father nears the end of his life, filmm...",Kirsten Johnson,Unknown,Movie
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",TV Show
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",TV Show
3,Jailbirds New Orleans,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",Unknown,Unknown,TV Show
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",TV Show


## inspecting if there any null columns 

In [41]:
rec_df.isnull().sum()

title          0
listed_in      0
description    0
director       0
cast           0
type           0
dtype: int64

## 🔧 Feature Engineering

We create a combined text representation for each title by merging:
- genres (`listed_in`)
- description
- director
- cast
- type

We repeat `listed_in` to give more weight to genre information.

In [42]:
rec_df["combined_features"] = (
    rec_df["listed_in"] + " " +
    rec_df["listed_in"] + " " +
    rec_df["listed_in"] + " " +   # more weight
    rec_df["description"] + " " +
    rec_df["director"] + " " +
    rec_df["cast"]
)
rec_df[["title", "combined_features"]].head()

,title,combined_features
0,Dick Johnson Is Dead,Documentaries Documentaries Documentaries As h...
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysterie..."
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act..."
3,Jailbirds New Orleans,"Docuseries, Reality TV Docuseries, Reality TV ..."
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ..."


## Load Embedding Model

We use the pretrained `all-MiniLM-L6-v2` sentence-transformer model.


In [43]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Generate Embeddings

Each title is converted into a dense semantic vector.
These vectors capture the meaning of the combined text representation.

In [44]:
embeddings = model.encode(
    rec_df["combined_features"].tolist(),
    show_progress_bar=True
)

embeddings.shape

Batches:   0%|          | 0/276 [00:00<?, ?it/s]

(8807, 384)

## Compute Similarity

We compute cosine similarity between all embedding vectors.
This gives a similarity score between every pair of Netflix titles.

In [45]:
cosine_sim = cosine_similarity(embeddings, embeddings)
cosine_sim.shape

(8807, 8807)

## Title-to-Index Mapping

We create a mapping from each title to its row index.
This allows fast lookup when the user provides a title.

In [46]:
indices = pd.Series(rec_df.index, index=rec_df["title"])
indices.head()

title
Dick Johnson Is Dead     0
Blood & Water            1
Ganglands                2
Jailbirds New Orleans    3
Kota Factory             4
dtype: int64

## Recommendation Function

Given a title, the function:
1. finds the title index
2. gets similarity scores
3. sorts them in descending order
4. removes the title itself
5. keeps only titles with the same type
6. returns the top 5 recommendations

In [47]:
def recommend(title, top_n=10):
    if title not in indices:
        return f"Title '{title}' not found in the dataset."
    
    idx = indices[title]
    title_type = rec_df.iloc[idx]["type"]
    
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the title itself
    sim_scores = sim_scores[1:]
    
    # Keep only titles with the same type
    filtered_scores = [
        (i, score) for i, score in sim_scores
        if rec_df.iloc[i]["type"] == title_type
    ]
    
    # Select top N
    filtered_scores = filtered_scores[:top_n]
    recommended_indices = [i[0] for i in filtered_scores]
    
    return rec_df[["title", "type"]].iloc[recommended_indices]

## testing the system
We test the recommendation system on a few Netflix titles.

In [83]:
recommend_titles("War Machine")

,title,type,listed_in,description,similarity_score
78,Tughlaq Durbar,Movie,"Comedies, Dramas, International Movies",A budding politician has devious plans to rise...,0.627817
79,Tughlaq Durbar (Telugu),Movie,"Comedies, Dramas, International Movies",A budding politician has devious plans to rise...,0.627817
6522,Company,Movie,"Action & Adventure, Dramas, International Movies","Following a misunderstanding, a gangster and h...",0.589351
3126,Bangistan,Movie,"Comedies, Dramas, International Movies",Sent on the same deadly international mission ...,0.574010
1122,Madam Chief Minister,Movie,"Dramas, International Movies",Rising from disadvantage to become a state lea...,0.565660


In [82]:
recommend_titles("Stranger Things")

,title,type,listed_in,description,similarity_score
8574,ThirTEEN Terrors,TV Show,"International TV Shows, TV Horror, TV Mysteries",A group of teens searches for the dark truth b...,0.713992
1335,The Sinner,TV Show,"Crime TV Shows, TV Dramas, TV Mysteries",When a young mother inexplicably stabs a stran...,0.697605
8421,The Messengers,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy",A group of strangers who are killed by a myste...,0.693082
3887,Chambers,TV Show,"TV Horror, TV Mysteries, Teen TV Shows",Haunted by eerie visions and sinister impulses...,0.688524
1701,American Horror Story,TV Show,"TV Horror, TV Mysteries, TV Thrillers",This twisted Emmy-winning drama plays upon the...,0.687951


In [51]:
recommend_titles("Ozark", top_n=5)

,title,type,listed_in,description,similarity_score
921,StartUp,TV Show,"Crime TV Shows, TV Dramas",An attempt to launder stolen money finances a ...,0.625289
5752,Spotless,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",The law-abiding owner of a crime scene cleanin...,0.563254
3235,Shot Caller,Movie,"Dramas, Thrillers","Trying to go straight, a once-successful busin...",0.540286
465,Heist,TV Show,"Crime TV Shows, Docuseries",Millions in stolen cash. Missing luxury bourbo...,0.538271
3361,Tunnel,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",A detective finds himself 30 years in the futu...,0.532988


In [79]:
for index, row in breaking_bad.iterrows():
    # You can access any column in the row here
    print(f"{row['title']} : {row['description']}")
    print()


Mortel : After making a deal with a supernatural figure, two high schoolers emerge with extraordinary powers and join forces to solve a murder.

Age of Rebellion : At their high school, a group of unruly teens wreak havoc, face bullies and navigate turbulent lives beyond school grounds.

American Vandal : A high school is rocked by an act of vandalism, but the top suspect pleads innocence and finds an ally in a filmmaker. A satirical true crime mystery.

Have You Ever Fallen in Love, Miss Jiang? : A new teacher finds herself in an unenviable situation after witnessing a troubling interaction between an administrator and a student.

How to Fix a Drug Scandal : Two drug lab chemists' shocking crimes cripple a state's judicial system and blur the lines of justice for lawyers, officials and thousands of inmates.



In [81]:
recommend_titles("Narcos")

,title,type,listed_in,description,similarity_score
1268,El final del paraíso,TV Show,"Crime TV Shows, International TV Shows, Spanis...","In Colombia, the DEA's new director targets a ...",0.772210
2921,Narcos: Mexico,TV Show,"Crime TV Shows, TV Action & Adventure, TV Dramas",Witness the birth of the Mexican drug war in t...,0.761271
6672,El Cartel,TV Show,"Crime TV Shows, International TV Shows, Spanis...",One man makes a fateful decision to get involv...,0.737970
6673,El Cartel 2,TV Show,"Crime TV Shows, International TV Shows, Spanis...",Drug trafficker Pepe Cadena navigates the trea...,0.725478
4750,El Chapo,TV Show,"Crime TV Shows, Spanish-Language TV Shows, TV ...",This drama series chronicles the true story of...,0.713879


## Conclusion

In this notebook, we built a recommendation system using semantic embeddings.

### Achievements
- Combined multiple metadata features into one text representation
- Generated semantic embeddings using a pretrained transformer model
- Computed similarity between titles
- Built a recommendation function for top 5 similar titles

### Limitations
- No user personalization
- No popularity or rating signal
- Recommendations depend only on metadata and text

### Future Improvements
- Add popularity-based ranking
- Compare against TF-IDF recommendations
- Build a hybrid recommender system